In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 3: Local and upstream. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_localup as L, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')
print('pairs in the catalog (local -> upstream), L6:', len(L.partners(6)))
print(sorted(L.partners(6).items()))


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505
pairs in the catalog (local -> upstream), L6: 14
[('aridity', 'aridity_upstream'), ('cropland_extent', 'cropland_extent_upstream'), ('human_footprint_09', 'human_footprint_09_upstream'), ('karst', 'karst_upstream'), ('pasture_extent', 'pasture_extent_upstream'), ('pct_clay', 'pct_clay_upstream'), ('pct_sand', 'pct_sand_upstream'), ('pct_silt', 'pct_silt_upstream'), ('precip_yr', 'precip_yr_upstream'), ('river_area', 'river_area_upstream'), ('slope_avg', 'slope_upstream'), ('temp_yr', 'temp_yr_upstream'), ('wet_pct_grp1', 'wet_pct_grp1_upstream'), ('wet_pct_grp2', 'wet_pct_grp2_upstream')]


In [2]:
# Cell 2 — The five Local-and-upstream statement types (rules in scripts/edop/dashboard/wo02_rules_localup.py)
for s in L.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print("an 'item' is a pair (local row + its _upstream row) or a continuous variable that has a local value only")


LU-01  Percentile gap, local versus upstream (one basin)
      applies to: single basin and one-basin buffer; the 13 variables that have an upstream pair; L6, L8
      template  : {name}: upstream ranks at the {ord} percentile and local at the {ord}, {gap} percentile points {higher|lower} upstream.
LU-02  Percentile gap, local versus upstream (area)
      applies to: buffer, polity; the 13 paired variables; L6, L8
      template  : {name}: across the {n} basins in this {area}, the upstream mean percentile is the {ord} and the local the {ord}, {gap} percentile points {higher|lower} upstream.
LU-03  Raw values, local versus upstream (one basin)
      applies to: single basin and one-basin buffer; the 13 paired variables
      template  : {name}: {raw}{ units} in this basin and {raw}{ units} across everything upstream.
LU-04  One side has no usable percentile
      applies to: all scopes with a paired variable
      template  : {name}: local {state}; upstream {state}.
LU-05  No upstream v

In [2]:
# Cell 3 — Fixtures: apply every rule to every item of the 33 fixtures; count where each applies and why not
fdf = A.apply_fixtures(fx, L)
items = fdf.groupby(['qid', 'variable']).paired.first()
print(f'items: {len(items)} ({int(items.sum())} pairs, {int((~items).sum())} local-only)')
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply (count of items):')
print(A.reason_table(fdf).to_string())


items: 896 (416 pairs, 480 local-only)
res                 applied  not_applicable
id    scope                                
LU-01 buffer              0             140
      polity              0             112
      single_basin      241             403
LU-02 buffer             23             117
      polity             51              61
      single_basin        0             644
LU-03 buffer              0             140
      polity              0             112
      single_basin      244             400
LU-04 buffer             42              98
      polity              1             111
      single_basin       58             586
LU-05 buffer             75              65
      polity             60              52
      single_basin      345             299

why a statement did not apply (count of items):
id     reason         
LU-01  multi_unit          74
       no_partner         480
       side_not_usable    101
LU-02  no_partner         480
       one_unit      

In [3]:
# Cell 4 — Draft sentences on chosen fixtures (up to 2 per statement type), to read and judge
EXAMPLES = ['F01', 'F04', 'F18', 'F22', 'F32']
for qid in EXAMPLES:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.head(2).iterrows():
            print(f"  {sid}  {r['text']}")
    print()


--- F01: arid; exogenous water (Niger); outside_active_domain
  LU-01  Aridity index (P/PET): upstream ranks at the 36th percentile and local at the 11th, 25 percentile points higher upstream.
  LU-01  Human footprint 2009: upstream ranks at the 72nd percentile and local at the 59th, 13 percentile points higher upstream.
  LU-03  Aridity index (P/PET): 9 P/PET ×100 in this basin and 47 P/PET ×100 across everything upstream.
  LU-03  Human footprint 2009: 63 in this basin and 86 across everything upstream.
  LU-04  Cropland extent: local is zero; upstream ranks at the 43rd percentile.
  LU-04  Karst area: local is zero; upstream is zero.
  LU-05  There is no separate upstream value for Flow distance to marine outlet.
  LU-05  There is no separate upstream value for Elevation maximum.

--- F04: large buffer; suppressed scores; section 2.6 reference counts
  LU-02  Clay content: across the 9 basins in this buffer, the upstream mean percentile is the 32nd and the local the 28th, 4 percenti

In [4]:
# Cell 5 — Population sample: how often each statement applies (share of items), weighted to the world and not
pdf = A.apply_population(pop, W, L)
print('items per statement type:', len(pdf) // len(L.STATEMENTS), '(13 pairs + 15 local-only variables per query)')
print('WEIGHTED by climate-zone size (percent of items):')
print((A.rates(pdf) * 100).round(1).to_string())
print()
print('UNWEIGHTED (percent of items):')
print((A.unweighted_rates(pdf) * 100).round(1).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3 for polity counts')


items per statement type: 14140 (13 pairs + 15 local-only variables per query)
WEIGHTED by climate-zone size (percent of items):
kind   buffer L6  single L6  single L8
id                                    
LU-01        0.1       34.9       33.0
LU-02       24.7        0.0        0.0
LU-03        0.1       35.7       33.7
LU-04       21.7       11.5       13.5
LU-05       53.6       53.6       53.6

UNWEIGHTED (percent of items):
kind   buffer L6  single L6  single L8
id                                    
LU-01        0.4       30.1       28.8
LU-02       21.8        0.0        0.0
LU-03        0.5       31.9       30.1
LU-04       24.3       16.3       17.6
LU-05       53.6       53.6       53.6

polities are not in the population sample (5 queries, all fixtures); see Cell 3 for polity counts


In [6]:
# Cell 6 — How big is the gap between local and upstream? (percentile points; positive = upstream higher)
print('absolute gap, weighted quantiles (both sides have a usable percentile):')
print(A.stat_quantiles(pdf, ['LU-01', 'LU-02']).round(1).to_string(index=False))
print()
gs = L.gap_summary(pdf)
print('by variable and query kind: weighted median signed gap, share with upstream higher, weighted median absolute gap:')
print(gs.round(2).to_string(index=False))


absolute gap, weighted quantiles (both sides have a usable percentile):
   id      kind    n  p10  p25  p50  p75  p90
LU-01 buffer L6   10  1.3  2.3  3.8 20.2 35.8
LU-01 single L6 2583  0.4  1.2  3.0  8.0 16.9
LU-01 single L8  806  0.3  1.0  2.7  6.9 14.8
LU-02 buffer L6  603  0.4  0.8  2.0  4.0  7.8

by variable and query kind: weighted median signed gap, share with upstream higher, weighted median absolute gap:
          variable      kind   n  median_signed  up_higher  median_abs
           aridity buffer L6  70           0.16       0.53        1.38
           aridity single L6 300          -0.13       0.46        1.36
           aridity single L8  99          -0.58       0.29        1.19
   cropland_extent buffer L6  29          -0.44       0.46        3.24
   cropland_extent single L6 116           0.89       0.56        3.35
   cropland_extent single L8  33           1.64       0.60        2.63
human_footprint_09 buffer L6  56          -0.27       0.44        1.74
human_footprint

In [5]:
# Cell 7 — Sweep: "agree" versus "differ" would need a cutoff on the gap. Evaluate a grid; do not choose
CUTS = [5, 10, 15, 20, 25]
print('LU-01 (one basin): weighted share of pairs whose gap is at least each cutoff, and share whose wording changes between 5 and 25 points:')
print(A.cutoff_sweep(pdf, 'LU-01', CUTS).round(3).to_string(index=False))
print()
print('LU-02 (area):')
print(A.cutoff_sweep(pdf, 'LU-02', CUTS).round(3).to_string(index=False))
print()
print('variables whose single-basin wording flips (count of pairs with a gap from 5 up to 25 points):')
print(A.flipping_variables(pdf, 'LU-01', CUTS[0], CUTS[-1]).to_string())


LU-01 (one basin): weighted share of pairs whose gap is at least each cutoff, and share whose wording changes between 5 and 25 points:
     kind    n  >= 5  >= 10  >= 15  >= 20  >= 25  flips
buffer L6   10 0.400  0.300  0.300  0.300  0.200  0.200
single L6 2583 0.364  0.200  0.120  0.079  0.053  0.310
single L8  806 0.342  0.169  0.098  0.039  0.029  0.313

LU-02 (area):
     kind   n  >= 5  >= 10  >= 15  >= 20  >= 25  flips
buffer L6 603 0.203  0.067  0.032  0.017  0.007  0.195

variables whose single-basin wording flips (count of pairs with a gap from 5 up to 25 points):
variable
pct_clay              141
pct_sand              116
pct_silt               96
slope_avg              95
human_footprint_09     79
precip_yr              69
karst                  60
pasture_extent         57


In [8]:
# Cell 8 — When a pair has no usable gap: what is each side? (weighted share of pairs, within each query kind; percent)
ct = L.state_crosstab(pdf)
print('rows = local state, columns = upstream state (usable = percentile above 0; zero = percentile 0; oad = outside_active_domain):')
print((ct * 100).round(1).to_string())
print()
d = pdf[(pdf.id == 'LU-04') & pdf.paired].copy()
d['local_empty_up_usable'] = d.lcode.isin(['oad', 'zero']) & (d.ucode == 'usable')
d['up_empty_local_usable'] = d.ucode.isin(['oad', 'zero']) & (d.lcode == 'usable')
for col in ['local_empty_up_usable', 'up_empty_local_usable']:
    print(col, '(weighted share of pairs, percent):', {k: round(float((g.w * g[col]).sum() / g.w.sum() * 100), 2) for k, g in d.groupby('kind')})
print()
chk = pdf[pdf.applies].groupby(['qid', 'variable']).id.apply(lambda s: tuple(sorted(s)))
print('every item gets exactly one of LU-01, LU-02, LU-04, LU-05 (LU-03 may accompany 01 or 04):')
print(chk.value_counts().to_string())


rows = local state, columns = upstream state (usable = percentile above 0; zero = percentile 0; oad = outside_active_domain):
ucode                 no_data   oad  suppressed  usable  zero
kind      lcode                                              
buffer L6 no_data         0.2   0.0         0.0     0.0   0.0
          oad             0.0   8.4         1.3     0.2   0.0
          suppressed      0.0   0.1        12.7     8.7   0.6
          usable          0.0   0.1        12.5    53.3   0.5
          zero            0.0   0.0         0.1     0.2   1.0
single L6 no_data         0.8   0.0         0.0     0.1   0.2
          oad             0.0  17.8         0.0     2.1   0.0
          usable          0.3   1.9         0.0    75.2   0.6
          zero            0.0   0.0         0.0     0.3   0.8
single L8 no_data         2.2   0.0         0.0     0.0   0.0
          oad             0.0  21.8         0.0     2.4   0.0
          usable          0.0   1.0         0.0    71.0   0.2
      

In [6]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Local and upstream rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
s1 = A.cutoff_sweep(pdf, 'LU-01', CUTS).set_index('kind')
s2 = A.cutoff_sweep(pdf, 'LU-02', CUTS).set_index('kind')
rows = []
for s in L.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    row = dict(s, family='Local and upstream')
    row['fixture_result'] = f"{int(f.applied.sum())} items applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = ('share of items, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.1%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.1%}" for k in ru.columns)
                                + '; polities not sampled')
    if sid == 'LU-01':
        row['sweep_result'] = ("an agree/differ label flips: the share of single-basin pairs with a gap of at least 5/10/15/20/25 points is "
                               + '/'.join(f"{s1.loc['single L6', f'>= {c}']:.0%}" for c in CUTS)
                               + f" (L6); {s1.loc['single L6', 'flips']:.0%} change wording between 5 and 25; gap stated as a number alone")
    elif sid == 'LU-02':
        row['sweep_result'] = ("an agree/differ label flips: the share of buffer pairs with a gap of at least 5/10/15/20/25 points is "
                               + '/'.join(f"{s2.loc['buffer L6', f'>= {c}']:.0%}" for c in CUTS)
                               + f"; {s2.loc['buffer L6', 'flips']:.0%} change wording between 5 and 25; gap stated as a number alone")
    else:
        row['sweep_result'] = 'no cutoff'
    rows.append(row)
n = P.upsert_master('Local and upstream', rows)
print('master table now has', n, 'rows ->', P.MASTER)


master table now has 18 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
